In [20]:
##Garantir o endereços de toda estrutura GOLD
from pathlib import Path

BASE = Path("../data/gold")
for path in BASE.rglob("*.parquet"):
    print(path)

..\data\gold\facts\fato_alfabetizacao_brasil\fato_alfabetizacao_brasil.parquet
..\data\gold\facts\fato_alfabetizacao_municipio\fato_alfabetizacao_municipio.parquet
..\data\gold\facts\fato_alfabetizacao_uf\fato_alfabetizacao_uf.parquet
..\data\gold\dimensions\dim_municipio\dim_municipio.parquet
..\data\gold\dimensions\dim_rede\dim_rede.parquet
..\data\gold\dimensions\dim_tempo\dim_tempo.parquet
..\data\gold\dimensions\dim_uf\dim_uf.parquet


## Explorando as dimensões

### Dimensão Municipio

In [3]:
import pandas as pd

df_municipio = pd.read_parquet("../data/gold/dimensions/dim_municipio/dim_municipio.parquet")

df_municipio.shape
#df_municipio.head(10)

(5548, 3)

In [27]:
df_municipio.head(10)

,id_municipio,Municipio,UF
0,5200050,Abadia de Goiás,GO
1,3100104,Abadia dos Dourados,MG
2,5200100,Abadiânia,GO
3,1500107,Abaetetuba,PA
4,3100203,Abaeté,MG
5,2300101,Abaiara,CE
6,2900207,Abaré,BA
7,4100103,Abatiá,PR
8,2900108,Abaíra,BA
9,4200051,Abdon Batista,SC


In [28]:
df_municipio.columns.tolist()

['id_municipio', 'Municipio', 'UF']

In [40]:
df_municipio.describe(include="all")

,id_municipio,Municipio,UF
count,5548,5548,5548
unique,5548,5279,27
top,5200050,Bom Jesus,MG
freq,1,5,853


In [39]:
df_municipio.isna().sum()

id_municipio    1
Municipio       1
UF              1
dtype: int64

In [40]:
df_municipio = df_municipio.dropna(subset=["UF"])

# Salvar novamente no parquet
df_municipio.to_parquet("../data/gold/dimensions/dim_municipio/dim_municipio.parquet", index=False)

In [41]:
df_municipio.isna().sum()

id_municipio    0
Municipio       0
UF              0
dtype: int64

In [42]:
df_municipio.duplicated().sum()

np.int64(0)

In [43]:
df_municipio["UF"].value_counts()

UF
MG    853
SP    642
RS    493
BA    417
PR    399
SC    295
GO    245
PI    224
PB    223
MA    217
PE    185
CE    184
RN    167
PA    144
MT    141
TO    139
AL    102
RJ     92
MS     79
ES     78
SE     75
AM     61
RO     53
AC     22
AP     16
DF      1
RR      1
Name: count, dtype: int64

Ao todo o Brasil possui 5.569 municípios, mas se imagina que nem todos possuem uma meta definida, justificando a divergência para os 5.548 municípios da base GOLD, mas resgatando os dados da camada BRONZE foi possível verificar que os dados são mantidos, não havendo nenhum ponto de atenção aqui.

Havia um municipio nulo que foi devidamente excluído

### Dimensão Estado

In [23]:
import pandas as pd

df_estado = pd.read_parquet("../data/gold/dimensions/dim_uf/dim_uf.parquet")

df_estado.shape

(28, 1)

In [4]:
df_estado.head(5)

,UF
0,AC
1,AL
2,AM
3,AP
4,BA


In [5]:
df_estado["UF"].value_counts()

UF
AC    1
AL    1
AM    1
AP    1
BA    1
CE    1
DF    1
ES    1
GO    1
MA    1
MG    1
MS    1
MT    1
PA    1
PB    1
PE    1
PI    1
PR    1
RJ    1
RN    1
RO    1
RR    1
RS    1
SC    1
SE    1
SP    1
TO    1
Name: count, dtype: int64

In [6]:
df_estado.describe(include="all")

,UF
count,27
unique,27
top,AC
freq,1


In [24]:
df_estado.isna().sum()

UF    1
dtype: int64

In [25]:
df_estado[df_estado["UF"].isna()]

,UF
27,NaN


In [26]:
# Remover linhas onde UF é nulo
df_estado = df_estado.dropna(subset=["UF"])

# Salvar novamente no parquet
df_estado.to_parquet("../data/gold/dimensions/dim_uf/dim_uf.parquet", index=False)

In [27]:
df_estado.isna().sum()

UF    0
dtype: int64

In [12]:
df_estado.duplicated().sum()

np.int64(0)

Dimensão estado possuia um dado nulo que foi descartado, pois não fazia sentido mantê-lo e não havia necessidade de um tratamento tão específico.

### Dimensão rede

In [28]:
import pandas as pd

df_rede= pd.read_parquet("../data/gold/dimensions/dim_rede/dim_rede.parquet")

df_rede.shape

(4, 2)

In [29]:
df_rede.head()

,ID_Rede,Rede
0,2,Estadual
1,3,Municipal
2,4,Privada
3,<NA>,NaN


In [30]:
df_rede = df_rede.dropna(subset=["Rede"])

df_rede.to_parquet("../data/gold/dimensions/dim_rede/dim_rede.parquet", index=False)

In [31]:
df_rede.head()

,ID_Rede,Rede
0,2,Estadual
1,3,Municipal
2,4,Privada


Tabela muito pequena, logo não há necessidade de exploração mais profunda, havia dado nulo que foi devidamente removido

### Dimensão Tempo

In [35]:
import pandas as pd

df_tempo = pd.read_parquet("../data/gold/dimensions/dim_tempo/dim_tempo.parquet")

df_tempo.shape

(2, 1)

In [36]:
df_tempo.head()

,ano
0,2023
1,2024


Sem necessidade de exploração ou limpeza

## Tabelas Fato

In [4]:
import pandas as pd

df_fact_br = pd.read_parquet("../data/gold/facts/fato_alfabetizacao_brasil/fato_alfabetizacao_brasil.parquet")

df_fact_br.shape

(5, 10)

In [5]:
df_fact_br.head()

,ano,ID_Rede,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento,meta
0,2023,2,155140,80662,131526,131522,51.99,84.78,84.78,NaN
1,2023,3,1592299,796765,1371532,1371287,50.04,86.14,86.12,NaN
2,2024,2,280258,150760,241173,241074,53.79,86.05,86.02,NaN
3,2024,3,1840277,956343,1611591,1610754,51.97,87.57,87.53,NaN
4,2024,4,25,16,24,24,64.00,96.00,96.00,NaN


In [47]:
import pandas as pd

df_fact_uf = pd.read_parquet("../data/gold/facts/fato_alfabetizacao_uf/fato_alfabetizacao_uf.parquet")

df_fact_uf.shape

(100, 11)

In [48]:
df_fact_uf.head()

,ano,UF,ID_Rede,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento,meta
0,2023,AC,2,61,42,53,53,68.85,86.89,86.89,NaN
1,2023,AC,3,239,92,181,181,38.49,75.73,75.73,NaN
2,2023,AL,2,1620,534,1434,1434,32.96,88.52,88.52,NaN
3,2023,AL,3,33756,13979,31231,31231,41.41,92.52,92.52,NaN
4,2023,AM,2,14041,7461,11810,11810,53.14,84.11,84.11,NaN


In [49]:
df_fact_uf.tail()

,ano,UF,ID_Rede,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento,meta
95,2024,SE,3,17276,6033,16006,15819,34.92,92.65,91.57,NaN
96,2024,SP,2,105973,63911,96546,96540,60.31,91.10,91.10,NaN
97,2024,SP,3,337815,167676,298945,298913,49.64,88.49,88.48,NaN
98,2024,TO,3,19688,8431,16764,16764,42.82,85.15,85.15,NaN
99,2024,TO,4,25,16,24,24,64.00,96.00,96.00,NaN


In [50]:
df_fact_uf.describe(include="all")

,ano,UF,ID_Rede,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento,meta
count,100.0,100,100.0,100.00000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,0.0
unique,<NA>,27,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,<NA>,AC,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,<NA>,4,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2023.52,NaN,2.54,38679.99000,19845.460000,33558.460000,33546.610000,50.336600,86.658100,86.606800,NaN
std,0.502117,NaN,0.520683,53388.45883,28553.992855,46672.150966,46666.849805,13.093206,6.624368,6.588981,NaN
min,2023.0,NaN,2.0,25.00000,16.000000,24.000000,24.000000,29.140000,61.360000,61.360000,NaN
25%,2023.0,NaN,2.0,1562.75000,625.000000,1376.250000,1376.250000,39.435000,83.052500,83.052500,NaN
50%,2024.0,NaN,3.0,19864.00000,7513.500000,16441.500000,16441.500000,48.740000,87.985000,87.900000,NaN
75%,2024.0,NaN,3.0,49723.50000,28506.250000,43906.500000,43906.500000,58.547500,90.525000,90.525000,NaN


### GOLD Municipios

### 1. Estrutura dos dados

In [6]:
import pandas as pd

df_fact_municipio = pd.read_parquet("../data/gold/facts/fato_alfabetizacao_municipio/fato_alfabetizacao_municipio.parquet")

df_fact_municipio.shape

(12431, 11)

In [8]:
df_fact_municipio.head(10)

,ano,id_municipio,ID_Rede,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento,meta
0,2023,1100015,3,254,153,227,227,60.24,89.37,89.37,NaN
1,2023,1100023,3,1361,764,1222,1222,56.14,89.79,89.79,NaN
2,2023,1100031,3,84,54,76,76,64.29,90.48,90.48,NaN
3,2023,1100049,2,227,132,200,200,58.15,88.11,88.11,NaN
4,2023,1100049,3,726,385,613,613,53.03,84.44,84.44,NaN
5,2023,1100056,3,241,127,222,222,52.70,92.12,92.12,NaN
6,2023,1100064,3,169,93,147,147,55.03,86.98,86.98,NaN
7,2023,1100072,3,129,71,119,119,55.04,92.25,92.25,NaN
8,2023,1100080,3,235,171,214,214,72.77,91.06,91.06,NaN
9,2023,1100098,2,107,67,96,96,62.62,89.72,89.72,NaN


In [9]:
df_fact_municipio.tail(10)

,ano,id_municipio,ID_Rede,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento,meta
12421,2024,5221601,3,310,235,305,305,75.81,98.39,98.39,71.02
12422,2024,5221700,3,132,129,132,132,97.73,100.00,100.00,76.25
12423,2024,5221809,3,37,23,34,34,62.16,91.89,91.89,78.38
12424,2024,5221858,3,2370,1346,2096,2094,56.79,88.44,88.35,60.56
12425,2024,5221908,3,46,39,46,46,84.78,100.00,100.00,80.00
12426,2024,5222005,3,164,137,156,153,83.54,95.12,93.29,80.00
12427,2024,5222054,3,95,80,95,95,84.21,100.00,100.00,78.50
12428,2024,5222203,3,61,48,61,61,78.69,100.00,100.00,80.00
12429,2024,5222302,3,73,52,65,65,71.23,89.04,89.04,75.29
12430,2024,5300108,2,27698,13140,22111,22111,47.44,79.83,79.83,NaN


In [7]:
df_fact_municipio.describe(include="all")

,ano,id_municipio,ID_Rede,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento,meta
count,12431.0,12431,12431.0,12431.000000,12431.000000,12431.000000,12431.000000,12431.000000,12431.000000,12431.000000,5232.000000
unique,<NA>,5548,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,<NA>,1100049,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,<NA>,4,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2023.526345,NaN,2.826884,311.157509,159.644920,269.957847,269.862521,55.968673,89.515630,89.487247,62.151745
std,0.499326,NaN,0.378575,1291.746277,651.467715,1099.000933,1098.720776,19.731870,8.647526,8.643910,15.098183
min,2023.0,NaN,2.0,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.940000
25%,2023.0,NaN,3.0,47.000000,25.000000,42.000000,42.000000,41.670000,85.830000,85.790000,52.170000
50%,2024.0,NaN,3.0,101.000000,54.000000,91.000000,91.000000,56.190000,91.030000,90.970000,64.095000
75%,2024.0,NaN,3.0,239.500000,128.000000,214.000000,214.000000,70.450000,95.120000,95.090000,75.885000


In [13]:
print("===== SHAPE =====")
print(df_fact_municipio.shape)

print("\n===== COLUNAS =====")
print(df_fact_municipio.columns.tolist())

print("\n===== TIPOS =====")
print(df_fact_municipio.dtypes)

print("\n===== NULOS =====")
print(df_fact_municipio.isna().sum())

print("\n===== ANOS =====")
print(sorted(df_fact_municipio["ano"].unique()))

print("\n===== REDES =====")
print(df_fact_municipio["ID_Rede"].value_counts().sort_index())

print("\n===== MUNICÍPIOS =====")
print(df_fact_municipio["id_municipio"].nunique())

===== SHAPE =====
(12431, 11)

===== COLUNAS =====
['ano', 'id_municipio', 'ID_Rede', 'alunos_avaliados', 'alunos_alfabetizados', 'alunos_presentes', 'provas_preenchidas', 'taxa_alfabetizacao', 'taxa_presenca', 'taxa_preenchimento', 'meta']

===== TIPOS =====
ano                       Int64
id_municipio                str
ID_Rede                   Int64
alunos_avaliados          int64
alunos_alfabetizados      int64
alunos_presentes          int64
provas_preenchidas        int64
taxa_alfabetizacao      float64
taxa_presenca           float64
taxa_preenchimento      float64
meta                    float64
dtype: object

===== NULOS =====
ano                        0
id_municipio               0
ID_Rede                    0
alunos_avaliados           0
alunos_alfabetizados       0
alunos_presentes           0
provas_preenchidas         0
taxa_alfabetizacao         0
taxa_presenca              0
taxa_preenchimento         0
meta                    7199
dtype: int64

===== ANOS =====
[np.i

In [34]:
df_fact_municipio[df_fact_municipio["meta"].notna() & (df_fact_municipio["ano"] == 2024)]["id_municipio"].nunique()


5232

In [35]:
df_fact_municipio[df_fact_municipio["meta"].notna() & (df_fact_municipio["ano"] == 2023)]["id_municipio"].nunique()

0

Confrotando com a base original, é possível garantir que todas as cidades que possuem meta definida para 2024 a Gold. Isso já é um ótimo indicativo, pois já temos insumo para usar a base de 2024 como teste dada a ausência de divergências. Além disso, 2023 não possui dados de meta, tanto na GOLD quanto na fonte de dados, logo podemos utilizar 2023 para treinamento.

In [15]:
chave = ["ano", "id_municipio", "ID_Rede"]

duplicados = (
    df_fact_municipio.groupby(chave)
      .size()
      .reset_index(name="qtd")
      .query("qtd > 1")
)

duplicados

,ano,id_municipio,ID_Rede,qtd


### Estatísticas da GOLD

In [16]:
df_fact_municipio["taxa_alfabetizacao"].describe()

count    12431.000000
mean        55.968673
std         19.731870
min          0.000000
25%         41.670000
50%         56.190000
75%         70.450000
max        100.000000
Name: taxa_alfabetizacao, dtype: float64

In [17]:
df_fact_municipio.groupby("ano").agg(
    taxa_media=("taxa_alfabetizacao", "mean"),
    taxa_mediana=("taxa_alfabetizacao", "median"),
    taxa_min=("taxa_alfabetizacao", "min"),
    taxa_max=("taxa_alfabetizacao", "max")
)

,taxa_media,taxa_mediana,taxa_min,taxa_max
ano,,,,
2023,54.740452,54.84,0.0,100.0
2024,57.073940,57.29,0.0,100.0


Os dados mostram que a taxa mediana teve um avanço de apenas 2,45 p.p YoY. 

In [22]:
df_fact_municipio["gap_meta"] = (
    df_fact_municipio["taxa_alfabetizacao"] - df_fact_municipio["meta"]
)

In [23]:
df_fact_municipio["gap_meta"].describe()

count    5232.000000
mean       -4.103979
std        15.407849
min       -68.890000
25%       -13.330000
50%        -4.445000
75%         5.650000
max        68.780000
Name: gap_meta, dtype: float64

Em média, o GAP dos munípios é de -4,1 p.p, com mediana de -4.4 p.p. Todavia, existe municípios totalmente fora da curva, podendo variar entre +68,8 p.p e -68,9 p.p

In [28]:
df_fact_municipio["atingiu_meta"] = (
    df_fact_municipio["taxa_alfabetizacao"] >= df_fact_municipio["meta"]
).where(
    df_fact_municipio["meta"].notna()
)

In [29]:
df_fact_municipio[df_fact_municipio["meta"].notna()]["atingiu_meta"].value_counts(
    normalize=True
)

atingiu_meta
False    0.628823
True     0.371177
Name: proportion, dtype: float64

In [30]:
df_fact_municipio.groupby("ano").agg(
    taxa_alfabetizacao=(
        "taxa_alfabetizacao",
        "mean"
    ),
    meta=(
        "meta",
        "mean"
    ),
    atingiu_meta=(
        "atingiu_meta",
        "mean"
    )
)

,taxa_alfabetizacao,meta,atingiu_meta
ano,,,
2023,54.740452,NaN,NaN
2024,57.073940,62.151745,0.371177


Os dados não parecem tão concentrados em uma única classificação, o que permite um bom treinamento.

In [31]:
df_2023 = df_fact_municipio[
    (df_fact_municipio["ano"] == 2023) &
    (df_fact_municipio["ID_Rede"] == 3)
].copy()

df_2024 = df_fact_municipio[
    (df_fact_municipio["ano"] == 2024) &
    (df_fact_municipio["ID_Rede"] == 3) &
    (df_fact_municipio["meta"].notna())
].copy()

print("Municípios 2023:", df_2023["id_municipio"].nunique())
print("Municípios 2024 com meta:", df_2024["id_municipio"].nunique())

ids_comuns = set(df_2023["id_municipio"]) & set(
    df_2024["id_municipio"]
)

print("Municípios com histórico 2023 + meta 2024:", len(ids_comuns))

Municípios 2023: 4825
Municípios 2024 com meta: 5232
Municípios com histórico 2023 + meta 2024: 4611


O alto volume de municípios com dados nos dois anos permite que utilizemos os dados de 2023 para treinamento e 2024 para teste, evitando Data leak

In [36]:
municipios_interseccao = df_fact_municipio[
    (df_fact_municipio["ano"] == 2024) &
    (df_fact_municipio["meta"].notna()) &
    (df_fact_municipio["ID_Rede"] == 3)
]["id_municipio"].unique()

df_municipio_interseccao = df_fact_municipio[
    (df_fact_municipio["ano"] == 2023) &
    (df_fact_municipio["ID_Rede"] == 3) &
    (df_fact_municipio["id_municipio"].isin(municipios_interseccao))
].copy()

In [37]:
df_municipio_interseccao["id_municipio"].nunique()

4611

In [38]:
pd.DataFrame({
    "coluna": df_municipio_interseccao.columns,
    "tipo": df_municipio_interseccao.dtypes.values,
    "nulos": df_municipio_interseccao.isna().sum().values,
    "nulos_%": (df_municipio_interseccao.isna().mean() * 100).round(2),
    "unicos": df_municipio_interseccao.nunique().values
})

,coluna,tipo,nulos,nulos_%,unicos
ano,ano,Int64,0,0.0,1
id_municipio,id_municipio,str,0,0.0,4611
ID_Rede,ID_Rede,Int64,0,0.0,1
alunos_avaliados,alunos_avaliados,int64,0,0.0,911
alunos_alfabetizados,alunos_alfabetizados,int64,0,0.0,658
alunos_presentes,alunos_presentes,int64,0,0.0,864
provas_preenchidas,provas_preenchidas,int64,0,0.0,862
taxa_alfabetizacao,taxa_alfabetizacao,float64,0,0.0,2756
taxa_presenca,taxa_presenca,float64,0,0.0,1640
taxa_preenchimento,taxa_preenchimento,float64,0,0.0,1636


In [40]:
df_municipio_interseccao[
    [
        "id_municipio",
        "alunos_avaliados",
        "alunos_alfabetizados",
        "alunos_presentes",
        "provas_preenchidas",
        "taxa_alfabetizacao",
        "taxa_presenca",
        "taxa_preenchimento"
    ]
].describe()

,alunos_avaliados,alunos_alfabetizados,alunos_presentes,provas_preenchidas,taxa_alfabetizacao,taxa_presenca,taxa_preenchimento
count,4611.000000,4611.000000,4611.000000,4611.000000,4611.000000,4611.000000,4611.000000
mean,332.335285,168.253524,288.192149,288.139015,54.880961,89.796764,89.788861
std,1224.279678,593.139725,1012.192620,1011.941824,19.558374,6.448345,6.447903
min,10.000000,1.000000,9.000000,9.000000,3.570000,70.000000,70.000000
25%,58.000000,30.000000,53.000000,53.000000,40.665000,85.900000,85.900000
50%,119.000000,62.000000,108.000000,108.000000,54.900000,90.520000,90.510000
75%,266.000000,141.000000,240.000000,239.500000,69.080000,94.695000,94.680000
max,54533.000000,25824.000000,44858.000000,44858.000000,100.000000,100.000000,100.000000


Dados de intersecção são bem representativos e confiáveis.